# DS675 Mini-Project: Diabetes Health Indicators
## Random Forest - Hyperparameter tunning and feature tunning




In [1]:
# imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, f1_score, recall_score,
                             precision_score, roc_auc_score, confusion_matrix,
                             classification_report, roc_curve)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
%matplotlib inline

## 0. Setup — load file.
### This is a binary, balanced version of the dataset

In [ ]:
#load file from drive.
from google.colab import drive
import os

# 1. Mount your Google Drive
# You will be prompted to grant permission the first time you run this
drive.mount('/content/drive')

# 2. Define the path (Option A: MyDrive with no space)
# Note: Ensure "Colab Notebooks" matches the exact spelling/capitalization in your Drive
csv_path = "/content/drive/MyDrive/Colab Notebooks/DS675/mini_project/data/diabetes_binary_5050split_health_indicators_BRFSS2015.csv"

# 3. Verify if the file exists at that path
if os.path.exists(csv_path):
    print("Success! File found.")
    # 4. Read the CSV
    df_full = pd.read_csv(csv_path)

    # Display the first 5 rows to confirm data loaded correctly
    #print(df.head())
else:
    print("File not found at the specified path.")
    print("Check your folder names for typos or try 'My Drive' with a space.")


Mounted at /content/drive
Success! File found.


##1. Train and Scale

In [ ]:
# split, scale
X = df_full.drop('Diabetes_binary', axis=1)
y = df_full['Diabetes_binary']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_std = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_std = pd.DataFrame(X_test_scaled, columns=X.columns)

## 2. Hyperparameter tuning for RF

In [ ]:
param_grid = {
    'n_estimators': [100, 150, 200, 300],
    'max_depth': [None, 10],
    'criterion' : ['gini', 'entropy'],
    'random_state':[42, 0]
}

rf = RandomForestClassifier()
grid_RF = GridSearchCV(estimator=rf, param_grid=param_grid, cv=3, scoring='accuracy')
grid_RF.fit(X_train_std, y_train)

GridSearchCV(cv=3, estimator=RandomForestClassifier(),
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_depth': [None, 10],
                         'n_estimators': [100, 150, 200, 300],
                         'random_state': [42, 0]},
             scoring='accuracy')

## 3. Results for the best RF
#### This is the best result found for RF for our dataset

In [ ]:
########################
#Read results
########################
results = pd.DataFrame(grid_RF.cv_results_)

# Filtering to the most important columns for readability
# Filtering to the most important columns for readability
relevant_columns = [
    'param_n_estimators',
    'param_max_depth',
    'param_criterion',
    'param_random_state',
    'mean_test_score',
    'rank_test_score'
]
all_results = results[relevant_columns].sort_values(by='rank_test_score')

print("--- Top 10 Parameter Combinations ---")
print(all_results.head(10))

print(f"\nBest Parameters: {grid_RF.best_params_}")
print(f"Best Cross-Validation Score: {grid_RF.best_score_:.4f}")

# 5. Use the best model to predict
best_RF = grid_RF.best_estimator_
final_predictions = best_RF.predict(X_test_std)

# 6. Evaluate

print(f"Accuracy: {accuracy_score(y_test, final_predictions):.2%}")
classificationreport = classification_report(y_test, final_predictions)
print(classificationreport)

--- Top 10 Parameter Combinations ---
    param_n_estimators param_max_depth param_criterion  param_random_state  \
13                 200              10            gini                   0   
11                 150              10            gini                   0   
30                 300              10         entropy                  42   
31                 300              10         entropy                   0   
27                 150              10         entropy                   0   
29                 200              10         entropy                   0   
25                 100              10         entropy                   0   
9                  100              10            gini                   0   
28                 200              10         entropy                  42   
26                 150              10         entropy                  42   

    mean_test_score  rank_test_score  
13         0.751596                1  
11         0.751578      

##4. Create datasets with 1 - 21 features.
####ordered by the importance given in SHAP analysis

In [ ]:
# Shap chose list of features
sorted_features = ['GenHlth', 'HighBP', 'BMI', 'Age', 'HighChol', 'Sex', 'Income', 'HeartDiseaseorAttack',
                   'DiffWalk', 'MentHlth', 'CholCheck', 'HvyAlcoholConsump', 'Education', 'PhysHlth', 'Stroke',
                   'Smoker', 'PhysActivity', 'Fruits', 'Veggies', 'AnyHealthcare', 'NoDocbcCost']
#####################################
# create data frames with k features
#####################################
X_trains = list(range(22))
for i in range(1, 22):
    X_trains[i] = X_train_std[sorted_features[0:i]].copy()

X_tests = list(range(22))
for i in range(1, 22):
    X_tests[i] = X_test_std[sorted_features[0:i]].copy()

##5. Fit the best RF to each of the partial models

In [ ]:
from sklearn.base import clone
######################
#feature counting
#Find the number of features needed for high accurracy
######################

#keep scores
accuracy_scores = list(range(22))

# fit the best modals to the partial datasets
for i in range(1, 22):
  # create a clone to make sure no data from last training attaches
  best_RF_p = clone(best_RF)
  best_RF_p.fit(X_trains[i], y_train)
  final_predictions = best_RF_p.predict(X_tests[i])

  #Results
  print("\n"+"*"*50)
  print(f"Number of features: {i}")
  accuracy_scores[i] = accuracy_score(y_test, final_predictions)
  print(f"Accuracy: {accuracy_score(y_test, final_predictions):.4%}")
  classificationreport = classification_report(y_test, final_predictions)
  print('\nClassification_report : ')
  print(classificationreport)
  print("\n"+"*"*50)



**************************************************
Number of features: 1
Accuracy: 67.9327%

Classification_report : 
              precision    recall  f1-score   support

         0.0       0.73      0.57      0.64      7070
         1.0       0.65      0.79      0.71      7069

    accuracy                           0.68     14139
   macro avg       0.69      0.68      0.68     14139
weighted avg       0.69      0.68      0.68     14139


**************************************************

**************************************************
Number of features: 2
Accuracy: 70.6203%

Classification_report : 
              precision    recall  f1-score   support

         0.0       0.70      0.72      0.71      7070
         1.0       0.71      0.69      0.70      7069

    accuracy                           0.71     14139
   macro avg       0.71      0.71      0.71     14139
weighted avg       0.71      0.71      0.71     14139


**************************************************

***

##6. How fast does the partial sets converge to the full set?

In [ ]:
#Find when we are preety close to the maximum accurecy
full_acc = accuracy_scores[-1]
found80 = found90 = found95 = False
acc_80 = acc_90 = acc_95 = 21

i = 1
while i < 22 and (not found80 or not found90 or not found95):
  if (accuracy_scores[i] / full_acc) > 0.8 and not found80:
    acc_80 = i
    found80 = True
  if (accuracy_scores[i] / full_acc) > 0.9 and not found90:
    acc_90 = i
    found90 = True
  if (accuracy_scores[i] / full_acc) > 0.95 and not found95:
    acc_95 = i
    found95 = True
  i = i+1

print(f"80% Number of features: {acc_80}")
print(f"Accuracy: {accuracy_scores[acc_80] / full_acc:.4%}")

print(f"90% Number of features: {acc_90}")
print(f"Accuracy: {accuracy_scores[acc_90] / full_acc:.4%}")

print(f"95% Number of features: {acc_95}")
print(f"Accuracy: {accuracy_scores[acc_95] / full_acc:.4%}")


80% Number of features: 1
Accuracy: 90.6645%
90% Number of features: 1
Accuracy: 90.6645%
95% Number of features: 3
Accuracy: 96.5924%


## 7. Conclusion

In [ ]:
features = sorted_features[0:acc_95]
print(f"KNN has {accuracy_scores[-1]:.2%} accuracy when considering all features." )
print(f"When considering just the features  { {*features} }  the accuracy is {accuracy_scores[acc_95]:.2%}")
print(f"which is {accuracy_scores[acc_95] / full_acc:.2%}  of the whole set accuracy")

KNN has 74.93% accuracy when considering all features.
When considering just the features  {'GenHlth', 'HighBP', 'BMI'}  the accuracy is 72.37%
which is 96.59%  of the whole set accuracy


In [ ]:
!jupyter nbconvert --to html "/content/drive/MyDrive/Colab Notebooks/DS675/mini_project/RFTuning.ipynb"


[NbConvertApp] Converting notebook /content/drive/MyDrive/Colab Notebooks/DS675/mini_project/RFTuning.ipynb to html
[NbConvertApp] Writing 346340 bytes to /content/drive/MyDrive/Colab Notebooks/DS675/mini_project/RFTuning.html
